# STEP 2. Table merge

Join the 5 tables, already narrowed in STEP 1 to 10 service categories and 13 quarters, at the commercial-area level.

## Join keys

| Right table | Key | Relationship |
|---|---|---|
| Area (area table) | `area_code` | Does not change over time, so no quarter |
| Foot traffic | `area_code` + `quarter_code` | |
| Attracting facilities | `area_code` + `quarter_code` | |
| Estimated sales | `area_code` + `quarter_code` + `service_code` | |

## Biggest risk in this step: row multiplication

If keys are duplicated in a right table, a left join alone can increase the row count. If a commercial area-quarter appears twice in the foot-traffic table, **every service-category row of that commercial area doubles**. It is very hard to trace the cause later.

There are two lines of defense.

1. **Before** - directly check that the right-table keys are unique before joining
2. **After** - double-check with the `validate=` argument and a row-count comparison

## Reflecting the STEP 1 overhaul

- The total store count column is **`store_count`**
- `area_total_stores` and the other area-level columns were **already computed in STEP 1**. They are not rebuilt here
- The food-service filter was also finished in STEP 1

In [1]:
import sys, os
from pathlib import Path

# Find the project root wherever the notebook is opened and add it to sys.path
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
print("Project root:", ROOT.name)

Project root: seoul-closure-risk


In [2]:
from config import PROC, QUARTERS, FOOD, MIN_STORES

store    = pd.read_pickle(PROC / "01_store.pkl")
sales    = pd.read_pickle(PROC / "01_sales.pkl")
area     = pd.read_pickle(PROC / "01_area.pkl")
flow     = pd.read_pickle(PROC / "01_flow.pkl")
facility = pd.read_pickle(PROC / "01_facility.pkl")

for nm, d in [("store", store), ("sales", sales), ("area", area),
              ("flow", flow), ("facility", facility)]:
    print(f"  {nm:9} {str(d.shape):>14}")

assert "store_count" in store.columns, "Re-run STEP 1 with the new version"
assert store["service_name"].nunique() == len(FOOD), "Service category filter was not applied"
print(f"\nQuarters {sorted(store['quarter_code'].unique())}")

  store       (160352, 14)
  sales         (87558, 5)
  area           (1650, 8)
  flow         (21434, 11)
  facility      (20514, 4)

Quarters [np.int64(20231), np.int64(20232), np.int64(20233), np.int64(20234), np.int64(20241), np.int64(20242), np.int64(20243), np.int64(20244), np.int64(20251), np.int64(20252), np.int64(20253), np.int64(20254), np.int64(20261)]


## 2-1. Key uniqueness check before the join

The standard approach is to stop row multiplication here. The post-join row-count comparison is only a backup.

In [3]:
def check_key(df, keys, name):
    dup = df.duplicated(keys).sum()
    status = "OK" if dup == 0 else f"❌ duplicates {dup:,}"
    print(f"  {name:12} {'+'.join(keys):40} {status}")
    assert dup == 0, f"{name} key duplicated - joining would multiply rows"

print("[Right-table key uniqueness]")
check_key(area,     ["area_code"],                                  "area")
check_key(flow,     ["quarter_code", "area_code"],                "foot traffic")
check_key(facility, ["quarter_code", "area_code"],                "facilities")
check_key(sales,    ["quarter_code", "area_code", "service_code"], "sales")

print("\n[Left-table key uniqueness]")
check_key(store, ["quarter_code", "area_code", "service_code"], "store (base)")

[Right-table key uniqueness]
  area         area_code                                OK
  foot traffic quarter_code+area_code                   OK
  facilities   quarter_code+area_code                   OK
  sales        quarter_code+area_code+service_code      OK

[Left-table key uniqueness]
  store (base) quarter_code+area_code+service_code      OK


## 2-2. Left join

`validate="many_to_one"` makes pandas raise an exception itself if the right-table key is not unique. Together with the row-count comparison, this gives two lines of defense.

In [4]:
n0 = len(store)
df = store.copy()

def join(right, on, name, how="left", validate="many_to_one"):
    global df
    before = len(df)
    df = df.merge(right, on=on, how=how, validate=validate)
    assert len(df) == before == n0, f"[{name}] row multiplication {before:,} -> {len(df):,}"
    print(f"  [{name:10}] rows kept {len(df):,}  OK")

join(area[["area_code", "area_type_name", "district_name",
           "area_size", "x_coord", "y_coord"]]
     .rename(columns={"area_type_name": "area_type",
                      "district_name": "district"}),
     "area_code", "area")

join(flow[["quarter_code", "area_code", "foot_traffic_total"]],
     ["quarter_code", "area_code"], "foot traffic")

join(facility[["quarter_code", "area_code", "attracting_facility_count", "subway_station_count"]],
     ["quarter_code", "area_code"], "facilities")

join(sales, ["quarter_code", "area_code", "service_code"],
     "sales", validate="one_to_one")

# Cross-check that area_type_name in the store table matches area_type from the area table
mismatch = (df["area_type_name"] != df["area_type"]).sum()
print(f"\narea_type mismatch between the two sources : {mismatch:,}")
df = df.drop(columns=["area_type_name"])
print(f"Join complete {df.shape}")

  [area      ] rows kept 160,352  OK
  [foot traffic] rows kept 160,352  OK
  [facilities] rows kept 160,352  OK
  [sales     ] rows kept 160,352  OK

area_type mismatch between the two sources : 0
Join complete (160352, 23)


## 2-3. Missing-value diagnosis

In [5]:
rows = []
for c in ["district", "area_size", "x_coord", "foot_traffic_total",
          "attracting_facility_count", "subway_station_count", "monthly_sales", "area_total_stores"]:
    r = df[c].isna().mean()
    verdict = "use as is" if r < 0.05 else ("caution" if r < 0.30 else "exclude from matching covariates")
    rows.append({"column": c, "missing_rate": f"{r:.1%}", "verdict": verdict})
display(pd.DataFrame(rows))

,column,missing_rate,verdict
0,district,0.0%,use as is
1,area_size,0.0%,use as is
2,x_coord,0.0%,use as is
3,foot_traffic_total,0.0%,use as is
4,attracting_facility_count,2.3%,use as is
5,subway_station_count,85.0%,exclude from matching covariates
6,monthly_sales,45.4%,exclude from matching covariates
7,area_total_stores,0.0%,use as is


## 2-4. What the missing sales really are ★

The 45% missing in `monthly_sales` was previously written off as "the sales file covers only some service categories", but **that is not true.** The sales file contains all 10 food-service categories.

The real cause is **cell-level de-identification**. For a commercial area x service category cell with only a few stores, publishing sales would reveal individual store sales, so the value is left blank. The table below shows that structure.

In [6]:
df["sales_missing_flag"] = df["monthly_sales"].isna()
bins = pd.cut(df["store_count"], [-1, 0, 1, 2, 4, 9, 19, 49, 10**6],
              labels=["0", "1", "2", "3-4", "5-9", "10-19", "20-49", "50+"])
t = (df.groupby(bins, observed=True)
     .agg(n_cells=("sales_missing_flag", "size"), sales_missing_rate=("sales_missing_flag", "mean")))
display(t.round(3))

sub = df[df["store_count"] >= MIN_STORES]
print(f"All                     missing sales {df['sales_missing_flag'].mean():.1%}")
print(f"Cells with {MIN_STORES}+ stores only  missing sales {sub['sales_missing_flag'].mean():.1%}")
print(f"Cells with {MIN_STORES}+ stores + neighborhood commercial area  missing sales "
      f"{sub[sub['area_type']=='neighborhood commercial area']['sales_missing_flag'].mean():.1%}")
print("\n-> STEP 3 filters out cells with fewer than 5 stores anyway,")
print("   sales are observed for over 90% of the analysis sample. The judgment that 'sales cannot be used' should be revisited.")

,n_cells,sales_missing_rate
store_count,,
0,1134,1.000
1,33812,1.000
2,22715,1.000
3-4,29766,0.335
5-9,31916,0.135
10-19,20472,0.033
20-49,14310,0.013
50+,6227,0.002


All                     missing sales 45.4%
Cells with 5+ stores only  missing sales 7.1%
Cells with 5+ stores + neighborhood commercial area  missing sales 9.1%

-> STEP 3 filters out cells with fewer than 5 stores anyway,
   sales are observed for over 90% of the analysis sample. The judgment that 'sales cannot be used' should be revisited.


In [7]:
print("[Cells with 5+ stores] Missing-sales rate by service category")
display(sub.groupby("service_name")["sales_missing_flag"]
        .mean().sort_values().round(3).rename("sales_missing_rate").to_frame())
print("Note: missingness differs by service category. If sales are put into the matching covariates,")
print("   the sample drops asymmetrically by service category, so it should be handled as a robustness analysis, not the main analysis.")
df = df.drop(columns=["sales_missing_flag"])

[Cells with 5+ stores] Missing-sales rate by service category


,sales_missing_rate
service_name,
Korean restaurants,0.012
Beer and casual pubs,0.014
Korean snack-food restaurants,0.053
Chinese restaurants,0.061
Coffee and beverages,0.073
Fried-chicken shops,0.090
Japanese restaurants,0.126
Fast-food restaurants,0.158
Bakeries,0.162


Note: missingness differs by service category. If sales are put into the matching covariates,
   the sample drops asymmetrically by service category, so it should be handled as a robustness analysis, not the main analysis.


## 2-5. Quarterly coverage

Check that all 13 quarters were joined and that missing values are not concentrated in particular quarters. If only a particular quarter stands out, it is a problem in the source file.

In [8]:
cov = df.groupby("quarter_code").agg(
    n_rows=("store_count", "size"),
    n_areas=("area_code", "nunique"),
    foot_traffic_missing=("foot_traffic_total", lambda s: s.isna().mean()),
    attracting_facility_missing=("attracting_facility_count", lambda s: s.isna().mean()),
    sales_missing_flag=("monthly_sales", lambda s: s.isna().mean()))
display(cov.round(3))

assert set(cov.index) == set(QUARTERS), "Quarter coverage differs from config.QUARTERS"
print(f"All {len(cov)} quarters present ✅")
display(df["area_type"].value_counts().to_frame("row_count"))

,n_rows,n_areas,foot_traffic_missing,attracting_facility_missing,sales_missing_flag
quarter_code,,,,,
20231,12397,1631,0.000,0.023,0.446
20232,12409,1631,0.000,0.023,0.448
20233,12393,1630,0.000,0.023,0.447
20234,12405,1631,0.000,0.023,0.448
20241,12418,1632,0.000,0.023,0.450
20242,12388,1632,0.000,0.023,0.452
20243,12348,1633,0.001,0.023,0.453
20244,12320,1633,0.000,0.023,0.455
20251,12304,1633,0.000,0.023,0.459


All 13 quarters present ✅


,row_count
area_type,
neighborhood commercial area,102767
developed commercial area,31378
traditional market,25435
tourist special zone,772


## 2-6. Save

In [9]:
df.to_pickle(PROC / "02_merged.pkl")
mb = (PROC / "02_merged.pkl").stat().st_size / 1024**2
print(f"[Saved] 02_merged.pkl  {df.shape}  {mb:.1f} MB")
print(f"Columns: {len(df.columns)}")
print(list(df.columns))

[Saved] 02_merged.pkl  (160352, 23)  26.2 MB
Columns: 23
['quarter_code', 'area_code', 'service_code', 'service_name', 'store_count', 'nonfranchise_store_count', 'franchise_store_count', 'opening_count', 'closure_count', 'area_total_stores', 'area_food_stores', 'food_service_share', 'franchise_share', 'area_type', 'district', 'area_size', 'x_coord', 'y_coord', 'foot_traffic_total', 'attracting_facility_count', 'subway_station_count', 'monthly_sales', 'monthly_sales_count']


---

## What to do in STEP 3

- `y(t) = [net_decline(t+1) < 0]` - `shift(-1)` is applied after sorting by quarter within each (commercial area, service category) group
- Because 20261 exists, **labels are attached up to 20254** -> training period of 12 quarters (20231 to 20254)
- The 20261 rows themselves cannot get a label and drop out - this is normal
- Filters: label exists -> `store_count` >= 5 -> foot traffic exists -> valid area size
- Because the total store count basis changed, **more cells pass the 5-or-more-stores filter than before** (especially franchise-dense service categories)